In [4]:
import json
import os
from datetime import datetime

ARCHIVO_DB = "autos_colombia_db.json"


def cargar_datos():
  if os.path.exists(ARCHIVO_DB):
    with open(ARCHIVO_DB, "r", encoding="utf-8") as f:
      datos = json.load(f)
  else:
    datos = {
        "VEHICULOS_ACTIVOS": {},
        "HISTORIAL DE MOVIMIENTOS": [],
        "NOVEDADES": [],
        "CELDAS": {},
        "USUARIOS": {},
        "PAGOS": [],
    }

  if "CELDAS" not in datos:
    datos["CELDAS"] = {}
  if "USUARIOS" not in datos:
    datos["USUARIOS"] = {}
  if "PAGOS" not in datos:
    datos["PAGOS"] = []

  return datos


def guardar_datos(datos):
  with open(ARCHIVO_DB, "w", encoding="utf-8") as f:
    json.dump(datos, f, indent=4, ensure_ascii=False)


def celda_ocupada(datos, celda):
  celda_norm = celda.strip().upper()
  for info in datos["VEHICULOS_ACTIVOS"].values():
    if info["celda"].upper() == celda_norm:
      return True
  return False


def registrar_entrada(datos, placa, tipo_vehiculo, celda):
  placa = placa.strip().upper()
  celda = celda.strip().upper()
  tipo_vehiculo = tipo_vehiculo.strip().capitalize()

  if not placa or not celda:
    print("ERROR!!!: La PLACA y la CELDA son OBLIGATORIAS.")
    return

  if placa in datos["VEHICULOS_ACTIVOS"]:
    print(f" La CELDA {celda} ya está ocupada por otro vehículo.")
    return

  celdas = datos.get("CELDAS", {})
  if celdas:
    if celda not in celdas:
      print(f"ERROR!!!: La celda {celda} no existe.")
      return
    if celdas[celda]["estado"] != "DISPONIBLE":
      print(
          f" La CELDA {celda} no está disponible (estado:"
          f" {celdas[celda]['estado']})."
      )
      return
  elif celda_ocupada(datos, celda):
    print(f" La CELDA {celda} ya está ocupada por otro vehículo.")
    return

  ahora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
  datos["VEHICULOS_ACTIVOS"][placa] = {
      "placa": placa,
      "tipo de vehiculo": tipo_vehiculo,
      "celda": celda,
      "fecha_hora_entrada": ahora,
  }

  if celda in celdas:
    datos["CELDAS"][celda]["estado"] = "OCUPADA"

  guardar_datos(datos)
  print(
      f" ENTRADA REGISTRADA: Vehiculo {placa} ({tipo_vehiculo}) en celda {celda}"
      f" a las {ahora}. "
  )


def registrar_novedad(datos, placa, descripcion):
  placa = placa.strip().upper()
  descripcion = descripcion.strip()

  if not placa or not descripcion:
    print("ERROR!!!: Porfavor, ingrese la placa y la descripcion de la novedad.")
    return

  ahora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
  novedad = {
      "placa": placa,
      "descripcion": descripcion,
      "fecha_hora": ahora,
      "estado_en_parqueadero": (
          "Dentro" if placa in datos["VEHICULOS_ACTIVOS"] else "No activo / Historial"
      ),
  }

  datos["NOVEDADES"].append(novedad)
  guardar_datos(datos)
  print(
      f"LA NOVEDAD YA FUE REGISTRADA PARA EL VEHICULO CON PLACA {placa}"
      f" ({ahora}): {descripcion}"
  )


def listar_vehiculos_activos(datos):
  activos = datos["VEHICULOS_ACTIVOS"]
  if not activos:
    print("NO HAY VEHICULOS ACTUALMENTE DENTRO DEL PARQUEADERO.")
    return

  print("\n VEHÍCULOS ACTUALMENTE EN AUTOS COLOMBIA:")
  print("-" * 60)
  for info in activos.values():
    print(
        f"Placa: {info['placa']} | Tipo: {info['tipo de vehiculo']} | "
        f"Celda: {info['celda']} | Entrada: {info['fecha_hora_entrada']}"
    )
  print("-" * 60)


def registrar_salida(datos, placa):
  placa = placa.strip().upper()

  if placa not in datos["VEHICULOS_ACTIVOS"]:
    print(f" No se encontró un vehículo activo con la placa {placa}.")
    return

  ingreso = datos["VEHICULOS_ACTIVOS"].pop(placa)
  dt_entrada = datetime.strptime(
      ingreso["fecha_hora_entrada"], "%Y-%m-%d %H:%M:%S"
  )
  dt_salida = datetime.now()
  permanencia = dt_salida - dt_entrada
  horas_aprox = round(permanencia.total_seconds() / 3600, 2)

  registro_cerrado = {
      **ingreso,
      "fecha_hora_salida": dt_salida.strftime("%Y-%m-%d %H:%M:%S"),
      "permanencia_horas": horas_aprox,
  }
  datos["HISTORIAL DE MOVIMIENTOS"].append(registro_cerrado)

  celda_lib = ingreso["celda"].strip().upper()
  if celda_lib in datos.get("CELDAS", {}):
    datos["CELDAS"][celda_lib]["estado"] = "DISPONIBLE"

  guardar_datos(datos)
  print(
      f"Salida registrada: Vehículo {placa} liberó la celda"
      f" {ingreso['celda']}. Permanencia aprox.: {horas_aprox} h."
  )


########## AQUI DARE INICIO CON  LA SEGUNDA PARTE: GESTION DE CELDAS #############


def inicializar_celdas(datos):
  if "CELDAS" not in datos:
    datos["CELDAS"] = {}

  if not datos["CELDAS"]:
    datos["CELDAS"] = {
        "A-01": {"codigo": "A-01", "tipo": "AUTOMOVIL", "estado": "DISPONIBLE"},
        "A-02": {"codigo": "A-02", "tipo": "AUTOMOVIL", "estado": "DISPONIBLE"},
        "M-01": {"codigo": "M-01", "tipo": "MOTO", "estado": "DISPONIBLE"},
    }
    guardar_datos(datos)


def listar_celdas(datos):
  celdas = datos.get("CELDAS", {})
  if not celdas:
    print("NO HAY MAS CELDAS DISPONIBLES/REGISTRADAS.")
    return

  print("ESTADO DE CELDAS EN AUTOS COLOMBIA:")
  print("-" * 55)
  for c in celdas.values():
    print(
        f"Celda: {c['codigo']} | Tipo: {c['tipo']} | Estado: {c['estado']}"
    )
  print("-" * 55)


def cambiar_estado_celda(datos, codigo, nuevo_estado):
  codigo = codigo.strip().upper()
  nuevo_estado = nuevo_estado.strip().upper()
  estados_validos = ["DISPONIBLE", "OCUPADA", "MANTENIMIENTO"]

  if codigo not in datos.get("CELDAS", {}):
    print(f"ERROR!!!: La celda {codigo} no existe. ")
    return

  if nuevo_estado not in estados_validos:
    print(f"ERROR!!!: Estado no valido. Use: {estados_validos} ")
    return

  datos["CELDAS"][codigo]["estado"] = nuevo_estado
  guardar_datos(datos)
  print(f"CELDA {codigo} ACTUALIZADA A ESTADO: {nuevo_estado}")


########## GESTION DE USUARIOS #############


def registrar_usuario(datos, cedula, nombre, telefono, placa=""):
  cedula = cedula.strip()
  nombre = nombre.strip().title()
  telefono = telefono.strip()
  placa = placa.strip().upper()

  if not cedula or not nombre:
    print("ERROR!!!: La CEDULA y el NOMBRE son OBLIGATORIOS.")
    return

  if cedula in datos["USUARIOS"]:
    print(f"ERROR!!!: El usuario con cedula {cedula} ya esta registrado.")
    return

  datos["USUARIOS"][cedula] = {
      "cedula": cedula,
      "nombre": nombre,
      "telefono": telefono,
      "placa": placa,
  }
  guardar_datos(datos)
  print(f"USUARIO REGISTRADO: {nombre} (CC: {cedula})")


def consultar_usuario(datos, cedula):
  cedula = cedula.strip()
  usuario = datos["USUARIOS"].get(cedula)
  if not usuario:
    print(f"No se encontro usuario con cedula {cedula}.")
    return

  print(
      f"Usuario: {usuario['nombre']} | CC: {usuario['cedula']} | "
      f"Tel: {usuario['telefono']} | Placa: {usuario['placa']}"
  )


def listar_usuarios(datos):
  usuarios = datos.get("USUARIOS", {})
  if not usuarios:
    print("NO HAY USUARIOS REGISTRADOS.")
    return

  print("\nUSUARIOS REGISTRADOS EN AUTOS COLOMBIA:")
  print("-" * 60)
  for u in usuarios.values():
    print(
        f"CC: {u['cedula']} | Nombre: {u['nombre']} | "
        f"Tel: {u['telefono']} | Placa: {u['placa']}"
    )
  print("-" * 60)


########## TERCERA PARTE: GESTION DE PAGOS #############


def registrar_pago(datos, placa, monto, metodo, concepto="Estadia"):
  placa = placa.strip().upper()
  metodo = metodo.strip().upper()
  concepto = concepto.strip().capitalize()

  if not placa or monto <= 0:
    print("ERROR!!!: Ingrese una PLACA valida y un MONTO mayor a cero.")
    return

  ahora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
  pago = {
      "placa": placa,
      "monto": monto,
      "metodo": metodo,
      "concepto": concepto,
      "fecha_hora": ahora,
  }
  datos["PAGOS"].append(pago)
  guardar_datos(datos)
  print(
      f"PAGO REGISTRADO: Placa {placa} | Monto: ${monto} | "
      f"Metodo: {metodo} ({ahora})"
  )


def listar_pagos(datos):
  pagos = datos.get("PAGOS", [])
  if not pagos:
    print("NO HAY PAGOS REGISTRADOS.")
    return

  print("\nHISTORIAL DE PAGOS EN AUTOS COLOMBIA:")
  print("-" * 65)
  total = 0
  for p in pagos:
    total += p["monto"]
    print(
        f"Placa: {p['placa']} | Concepto: {p['concepto']} | "
        f"Monto: ${p['monto']} | Metodo: {p['metodo']} | Fecha: {p['fecha_hora']}"
    )
  print("-" * 65)
  print(f"TOTAL RECAUDADO: ${total}")


# ---- PRUEVBA RAPIDA ----

db = cargar_datos()
inicializar_celdas(db)
registrar_usuario(db, "1017123456", "Carlos Perez", "3001234567", "ABC123")
registrar_entrada(db, "ABC123", "AUTOMOVIL", "A-01")
registrar_novedad(
    db, "ABC123", "POSIBLE RAYON LEVE EN LA PUERTA TRASERA DERECHA AL INGRESAR."
)
registrar_pago(db, "ABC123", 8000, "EFECTIVO", "Estadia")
listar_vehiculos_activos(db)
cambiar_estado_celda(db, "A-02", "MANTENIMIENTO")
listar_celdas(db)
listar_usuarios(db)
listar_pagos(db)

USUARIO REGISTRADO: Carlos Perez (CC: 1017123456)
 La CELDA A-01 ya está ocupada por otro vehículo.
LA NOVEDAD YA FUE REGISTRADA PARA EL VEHICULO CON PLACA ABC123 (2026-11-22 01:50:15): POSIBLE RAYON LEVE EN LA PUERTA TRASERA DERECHA AL INGRESAR.
PAGO REGISTRADO: Placa ABC123 | Monto: $8000 | Metodo: EFECTIVO (2026-11-22 01:50:15)

 VEHÍCULOS ACTUALMENTE EN AUTOS COLOMBIA:
------------------------------------------------------------
Placa: ABC123 | Tipo: Automovil | Celda: A-01 | Entrada: 2026-11-22 00:16:23
------------------------------------------------------------
CELDA A-02 ACTUALIZADA A ESTADO: MANTENIMIENTO
ESTADO DE CELDAS EN AUTOS COLOMBIA:
-------------------------------------------------------
Celda: A-01 | Tipo: AUTOMOVIL | Estado: DISPONIBLE
Celda: A-02 | Tipo: AUTOMOVIL | Estado: MANTENIMIENTO
Celda: M-01 | Tipo: MOTO | Estado: DISPONIBLE
-------------------------------------------------------

USUARIOS REGISTRADOS EN AUTOS COLOMBIA:
--------------------------------------